# Lab 10: Theme Discovery

**Business question:** Which themes in many comments deserve managerial attention?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | The 23 customer messages with text | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | CFPB complaint extract, `train` split (6,000 complaints) | about 75–90 min (Exercise 1 means reading complaints for each theme) | Three exercises and the interpretation |
| 3 Challenge | CFPB complaints | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

This lab is independent of **Discussion 2**. For your Discussion 2 post you use the Discussion 2 handout, the Module 10 video and the live session; you do not need to run the lab first.

In earlier labs the categories were fixed in advance: products, aspects, routing classes. Here nobody has written the categories yet. You let **themes** emerge from the text with two methods: **NMF** on TF-IDF features and **k-means clusters** of the saved sentence vectors from Module 5. Then you do the part that matters most. You read the documents behind each theme and label it. You test whether it survives a small change. And you decide what its count can and cannot say. No model is downloaded and no LLM is called.

## Part 1 Guided core

### 1.1 Import the libraries

Select the `ban6403` kernel in VS Code, then run the cell below.

In [ ]:
# Path builds file paths that work on Windows, Mac and Linux.
from pathlib import Path
import numpy as np
import pandas as pd
# TF-IDF features (Module 4), the NMF theme model and k-means clustering.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans

### 1.2 Set the data and output folders

The notebook runs from the `notebooks` folder, so the course data is one folder up, in `data`. Your saved files go to `outputs/m10`.

In [ ]:
# ".." means one folder up; the / operator joins folder and file names.
DATA = Path("../data")
OUTPUT_DIR = Path("../outputs/m10")
# parents=True also creates the outputs folder if needed; exist_ok=True means no error if the folder exists.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 1.3 Load the customer messages

We use the 23 customer messages from Module 1 that have text (FB022 is empty). They are made up and very short. They are too few for real themes, but small enough to see how the method works and to check every result by reading.

In [ ]:
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
# Drop the message with no text; reset_index numbers the rows 0, 1, 2, ... again.
feedback = feedback.dropna(subset=["text"]).reset_index(drop=True)
print("messages:", len(feedback))
feedback[["feedback_id", "text"]].head(6)

### 1.4 TF-IDF features

As in Module 4, each message becomes a row of TF-IDF weights, one column per word. We drop common English stop words such as *the* and *is*, so the themes are built from content words.

In [ ]:
# Same TF-IDF setup as Module 4; stop_words="english" drops common words such as "the".
fb_vectorizer = TfidfVectorizer(stop_words="english")
# fit_transform learns the vocabulary and returns one row of weights per message (a sparse matrix).
X_fb = fb_vectorizer.fit_transform(feedback["text"])
# The words behind the columns, in column order.
fb_terms = fb_vectorizer.get_feature_names_out()
print("TF-IDF matrix:", X_fb.shape)

### 1.5 Fit a small NMF model

**NMF** (non-negative matrix factorization) splits the TF-IDF matrix into two smaller tables of numbers that are never negative:

- `W` has one row per message and one column per theme: how much each theme is present in the message.
- `H` has one row per theme and one column per word: how strongly each word belongs to the theme.

A **theme** is therefore a weighted word list, plus the messages that use it. You choose the number of themes. NMF starts from a random guess, so we fix the seed with `random_state` to get the same result every time.

In [ ]:
# n_components=4 themes; init="random" starts from random numbers and random_state=0 fixes them.
# max_iter=1000 gives the fit enough rounds to settle.
fb_nmf = NMF(n_components=4, init="random", random_state=0, max_iter=1000)
# fit_transform fits the model and returns W; the fitted model keeps H in components_.
W_fb = fb_nmf.fit_transform(X_fb)
H_fb = fb_nmf.components_
print("W (messages x themes):", W_fb.shape, "| H (themes x words):", H_fb.shape)
# The theme weights of the first four messages, rounded to 2 decimals.
pd.DataFrame(W_fb, index=feedback["feedback_id"]).round(2).head(4)

### 1.6 Top terms for each theme

The top terms of a theme are the words with the largest weights in its row of `H`. This helper returns them for every theme. You will reuse it in Practice.

In [ ]:
def top_terms(H, terms, n=10):
    """Take H and the word list; return a list with the n top words of each theme."""
    result = []
    for row in H:
        # argsort returns positions from smallest to largest value;
        # sorting -row puts the largest weights first, and [:n] keeps the first n.
        top_positions = np.argsort(-row)[:n]
        result.append(list(terms[top_positions]))
    return result

# enumerate gives the theme number k next to each word list.
for k, words in enumerate(top_terms(H_fb, fb_terms, n=5)):
    print(f"theme {k}:", " ".join(words))

### 1.7 Representative messages

Top terms are only half of the evidence. The other half is the messages with the highest weight for each theme, called **representative documents**. We also give each message its **dominant theme**: the theme with its highest weight.

Some messages share almost no words with any theme. Their weights are all close to zero, so "highest weight" means nothing. We give them theme `-1` (no theme) when their highest weight is below a small cutoff, 0.01. In short texts this happens often: FB009 ("ear cushions") uses words that no theme is built on.

In [ ]:
NO_THEME_CUTOFF = 0.01
# For each message (row of W): its largest weight, and the theme (column) where that weight sits.
top_weight = W_fb.max(axis=1)
top_theme = W_fb.argmax(axis=1)
# np.where takes the top theme when the weight reaches the cutoff, and -1 otherwise.
feedback["theme"] = np.where(top_weight >= NO_THEME_CUTOFF, top_theme, -1)
no_theme_ids = feedback.loc[feedback["theme"] == -1, "feedback_id"]
print("messages with no theme:", ", ".join(no_theme_ids))

for k in range(4):
    # W_fb[:, k] is the theme-k column; the 3 positions with the highest weight are the representative messages.
    top_rows = np.argsort(-W_fb[:, k])[:3]
    n_messages = (feedback["theme"] == k).sum()
    print(f"theme {k} ({n_messages} messages):")
    for i in top_rows:
        # :.2f prints the weight with 2 decimals.
        print(f"   {feedback.loc[i, 'feedback_id']}  weight {W_fb[i, k]:.2f}  {feedback.loc[i, 'text']}")

### 1.8 Repeated text can make its own theme

Find the theme whose strongest messages include FB005 and FB020 (in my run it is theme 0). Print the two messages side by side; the `theme` column shows their theme number on your computer.

In [ ]:
# isin keeps the rows whose ID is in the list.
feedback.loc[feedback["feedback_id"].isin(["FB005", "FB020"]), ["feedback_id", "theme", "text"]]

The two messages are identical. Repeated text gives the same words high weight twice, so NMF can spend a whole theme on it. One customer copying a message, or many customers using the same template letter, can look like a recurring issue. In the complaints you will meet the same effect with template letters and with redaction marks such as `XXXX`.

### 1.9 Embedding clusters

The second method uses the sentence vectors (embeddings) of the same messages, saved in Module 5 in the same order. **k-means** puts the vectors into k groups called **clusters**. Each message belongs to the cluster whose center is closest. Like NMF, k-means starts from random centers, so we fix the seed.

The crosstab compares the two methods: rows are NMF themes, columns are clusters.

In [ ]:
# The saved vectors: one row per message, in the same order as the rows of feedback.
# The file stores small 16-bit numbers to save space; astype("float32") converts them to 32-bit numbers for k-means.
fb_vectors = np.load(DATA / "embeddings/feedback_minilm.npy").astype("float32")
# n_clusters=4 groups; n_init=10 tries 10 random starts and keeps the best one; random_state fixes the starts.
fb_kmeans = KMeans(n_clusters=4, n_init=10, random_state=0).fit(fb_vectors)
# labels_ holds the cluster number of each message.
feedback["cluster"] = fb_kmeans.labels_
# Count the messages for each pair of NMF theme (rows) and cluster (columns).
pd.crosstab(feedback["theme"], feedback["cluster"])

Row `-1` holds the messages with no NMF theme. k-means always assigns every message to a cluster, even a message that fits none well.

The two methods group the messages differently. NMF needs shared words. Embeddings can group messages that mean similar things in different words. Neither grouping is "correct": both are drafts for a person to read.

In [ ]:
# The message IDs in each cluster.
for c in range(4):
    cluster_ids = feedback.loc[feedback["cluster"] == c, "feedback_id"]
    print(f"cluster {c}:", ", ".join(cluster_ids))

### 1.10 Does the theme survive a refit?

A theme that disappears when you change the seed is weak evidence. Refit NMF with `random_state=1` and compare the top terms.

The helper `match_themes` finds, for each theme in the first fit, the theme in the second fit that shares the most top terms. It checks each theme on its own, so two themes can pick the same best match: it is not a one-to-one pairing. You will reuse it in Practice.

In the table, `shared_terms` 5 means the theme came back with all 5 of its top terms; a small number means it changed.

In [ ]:
def match_themes(H_a, H_b, terms, n=10):
    """Take two H tables; return a table with, for each theme of H_a, its best match in H_b and the shared top terms."""
    words_a = top_terms(H_a, terms, n)
    words_b = top_terms(H_b, terms, n)
    rows = []
    for k, words in enumerate(words_a):
        # Count the top terms this theme shares with each theme of the second fit.
        # set(a) & set(b) keeps the words that are in both lists.
        shared = []
        for other in words_b:
            shared.append(len(set(words) & set(other)))
        # argmax gives the position of the largest count (the first one if there is a tie).
        best = int(np.argmax(shared))
        rows.append({"theme": k, "best_match": best, "shared_terms": max(shared),
                     "top_terms": " ".join(words), "match_terms": " ".join(words_b[best])})
    return pd.DataFrame(rows)

# Same model, only the seed changes.
fb_nmf_seed1 = NMF(n_components=4, init="random", random_state=1, max_iter=1000)
fb_nmf_seed1.fit(X_fb)
match_themes(H_fb, fb_nmf_seed1.components_, fb_terms, n=5)

### 1.11 Label the themes

A label is a short name that a manager understands. Write it from the top terms **and** the representative messages, not from the terms alone. The labels below are ours; read the messages in 1.7 and decide whether you agree.

The numbers in `fb_labels` are the theme numbers I got. Before you run the cell, compare each label with the top terms (1.6) and messages (1.7) on your screen. If your numbers differ, move each label to the theme whose words and messages fit it.

In [ ]:
fb_labels = {0: "Refund arrived (one repeated message)",
             1: "Send a missing part or troubleshooting steps",
             2: "Needs help with a device",
             3: "Waiting for an order or a charge to be fixed"}
# Build the table one theme (row) at a time.
fb_top_words = top_terms(H_fb, fb_terms, n=5)
theme_rows = []
for k in range(4):
    theme_rows.append({"theme": k,
                       "label": fb_labels[k],
                       "top_terms": " ".join(fb_top_words[k]),
                       "messages": (feedback["theme"] == k).sum()})
fb_themes = pd.DataFrame(theme_rows)
fb_themes

### 1.12 Save your outputs

This writes two small CSV files to `outputs/m10`. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
feedback[["feedback_id", "theme", "cluster", "text"]].to_csv(OUTPUT_DIR / "feedback_themes.csv", index=False)
fb_themes.to_csv(OUTPUT_DIR / "feedback_theme_labels.csv", index=False)
print("saved: outputs/m10/feedback_themes.csv and outputs/m10/feedback_theme_labels.csv")

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. The theme built mostly from FB005 and FB020 (theme 0 in my run) looks like a recurring issue. Should a manager treat it as one? Why or why not?
2. Which theme changed when you refit with `random_state=1` (step 1.10)? Name it by its top terms, since theme numbers can differ on your computer. What does that tell you about labeling a theme after only one fit?

**Your answer:**

## Part 2 Practice: themes in real complaints

Now you look for themes in the 6,000 complaints of the `train` split. All fitting uses `train` only, so `valid` and `test` stay unseen for later work.

Keep two facts about this data in mind:
- The course dataset has a **fixed number of complaints per product** (for example 1,500 credit reporting and 750 mortgage complaints in `train`).
- Repeated **template letters** were thinned out: most copies of repeated openings were removed, but some near-copies remain.

### 2.1 Load the complaints

In [ ]:
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
# Keep the train split; reset_index numbers its rows 0, 1, 2, ... so they line up with the rows of W later.
train = complaints[complaints["split"].eq("train")].reset_index(drop=True)
print("train complaints:", len(train))
train["product"].value_counts()

### 2.2 A first try: redaction marks take over

CFPB replaces personal details with `XXXX` and dates with `XX/XX/XXXX` or `XX/XX/year>`. Fit NMF with English stop words only and look at the top terms. Expect this step to take a few seconds.

In [ ]:
# min_df=5: keep words that appear in at least 5 complaints.
raw_vectorizer = TfidfVectorizer(stop_words="english", min_df=5)
X_raw = raw_vectorizer.fit_transform(train["narrative"])
raw_terms = raw_vectorizer.get_feature_names_out()
raw_nmf = NMF(n_components=8, init="random", random_state=0, max_iter=1000)
W_raw = raw_nmf.fit_transform(X_raw)
# Dominant theme of each complaint: the column of W with the largest weight.
raw_theme = W_raw.argmax(axis=1)
for k, words in enumerate(top_terms(raw_nmf.components_, raw_terms)):
    # (raw_theme == k) is True/False per complaint; sum() counts the True values.
    n_complaints = (raw_theme == k).sum()
    print(f"theme {k} ({n_complaints} complaints):", " ".join(words))

Two of the eight themes are built from `xx`, `xxxx`, `year` and `00`. These are redaction marks and number pieces, not words a customer chose. Almost a quarter of the complaints have one of these two as their dominant theme.

### 2.3 Remove redaction marks before fitting

We take the redactions out of the text with a **regular expression** (a text pattern). It removes redacted dates (`XX/XX/year>`), any run of two or more capital X, and money amounts in braces such as `{$450.00}`. The vectorizer then keeps only words of two or more letters, so numbers are dropped too. `max_df=0.5` drops words that appear in more than half of the complaints, such as *credit* and *account*. They are too common to separate themes.

In [ ]:
# Three patterns joined by | ("or"): redacted dates, two or more capital X, and money amounts in braces.
REDACTION = r"XX/XX/year>|X{2,}|\{\$[\d,.]+\}"
# Replace every match with a space; regex=True reads REDACTION as a pattern, not as plain text.
train["theme_text"] = train["narrative"].str.replace(REDACTION, " ", regex=True)
print("before:", train.loc[0, "narrative"][:150])
print("after: ", train.loc[0, "theme_text"][:150])

# token_pattern keeps words of two or more letters (no numbers);
# min_df=5 keeps words in at least 5 complaints; max_df=0.5 drops words in more than half of them.
vectorizer = TfidfVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{2,}\b", min_df=5, max_df=0.5)
X = vectorizer.fit_transform(train["theme_text"])
terms = vectorizer.get_feature_names_out()
print("TF-IDF matrix:", X.shape)

### 2.4 Fit the NMF baseline: eight themes

This is the **baseline**: the first, simple model that later choices are compared with. We use 8 themes and seed 0. Almost every complaint has a clear top weight, so here we use the largest weight directly as the dominant theme.

The theme numbers below are the ones I got. On another computer or scikit-learn version, the same themes can get different numbers. Always find a theme by its top terms, not by its number. The helper `find_theme` does this: it returns the theme in which a given word has the largest weight.

In [ ]:
nmf = NMF(n_components=8, init="random", random_state=0, max_iter=1000)
W = nmf.fit_transform(X)
H = nmf.components_
train["theme"] = W.argmax(axis=1)
# How many complaints have almost no theme (largest weight below 0.01)?
n_weak = (W.max(axis=1) < 0.01).sum()
print("complaints with top weight below 0.01:", n_weak, "of", len(train))

# One row per theme: its number, its size and its top 10 terms.
baseline_top = top_terms(H, terms)
theme_rows = []
for k in range(8):
    theme_rows.append({"theme": k,
                       "complaints": (train["theme"] == k).sum(),
                       "top_terms": " ".join(baseline_top[k])})
theme_table = pd.DataFrame(theme_rows)

def find_theme(H, terms, word):
    """Take H, the word list and one word; return the theme in which that word has the largest weight."""
    # The word's column in H, then the row (theme) with the largest value in that column.
    column = list(terms).index(word)
    return int(np.argmax(H[:, column]))

theme_table

### 2.5 Read the representative complaints

The helper prints the complaints with the highest weight for one theme. By default it shows the first 300 characters, which is enough to browse. With `chars=None` it prints each complaint in full. Before you settle on a label, read the three representative complaints in full: an opening can show a template and hide the actual problem or remedy. Ask: do these complaints share a problem, or only some words?

In [ ]:
def show_theme(k, n=3, chars=300):
    """Take a theme number; print the n complaints with the highest weight for it (first chars characters, or all with chars=None)."""
    for i in np.argsort(-W[:, k])[:n]:
        print(f"{train.loc[i, 'complaint_id']} | {train.loc[i, 'product']} | weight {W[i, k]:.3f}")
        text = train.loc[i, "narrative"]
        # chars=None means: print the whole complaint.
        if chars is not None:
            text = text[:chars]
        # Line breaks inside a complaint become spaces; the final "\n" adds an empty line.
        print("   ", text.replace("\n", " "), "\n")

cash_theme = find_theme(H, terms, "cash")
print("Cash App theme:", cash_theme)
# Full text of the three representative complaints.
show_theme(cash_theme, chars=None)

The Cash App theme is about money that is held or missing, and support that does not fix it. Now read the largest theme (we find it by the word *told*). Its top terms (*bank, money, told, called, said*) describe the steps of a story, not one problem. Large themes like this often collect complaints that only share a way of writing.

In [ ]:
story_theme = find_theme(H, terms, "told")
print("Largest theme:", story_theme)
show_theme(story_theme)

### 2.6 Template letters

Some consumers, or services that write letters for them, use a template. Even after thinning, template wording can form its own theme. Look at the beginnings of the five strongest complaints in the theme whose top terms include *opened* and *balance*.

In [ ]:
form_theme = find_theme(H, terms, "opened")
print("Form-letter theme:", form_theme)
# The five complaints with the highest weight for this theme, first 250 characters each.
for i in np.argsort(-W[:, form_theme])[:5]:
    print(train.loc[i, "complaint_id"], "|", train.loc[i, "product"])
    print("   ", train.loc[i, "narrative"][:250], "\n")

Most of them are letters that list accounts with fields such as *Date Opened* and *Balance*. The theme describes a **way of writing**, not a separate customer problem. Label it as a form letter and do not count it as its own issue without reading more.

### 2.7 Themes by product

Each theme draws most of its complaints from one or two products. Remember that the number of complaints per product was fixed when the course dataset was built.

In [ ]:
# margins=True adds the row and column totals ("All").
pd.crosstab(train["theme"], train["product"], margins=True)

### 2.8 Embedding clusters

Now the comparison method. We run k-means with 8 clusters on the sentence vectors of the same 6,000 complaints. The vectors in `data/embeddings/cfpb_minilm.npy` were made with the same model as in Module 5. They are stored in the same row order as the complaints file, so we can pick the `train` rows directly.

To describe a cluster in words, we average the TF-IDF rows of its complaints and list the words with the largest averages. As with themes, cluster numbers can differ on another computer, so find clusters by their words.

In [ ]:
# Same conversion to 32-bit numbers as in 1.9.
cfpb_vectors = np.load(DATA / "embeddings/cfpb_minilm.npy").astype("float32")
# The vector rows follow the complaints file, so a True/False array for "train" picks the train rows.
is_train = complaints["split"].eq("train").to_numpy()
train_vectors = cfpb_vectors[is_train]

kmeans = KMeans(n_clusters=8, n_init=10, random_state=0).fit(train_vectors)
train["cluster"] = kmeans.labels_

# Average TF-IDF row of each cluster.
# to_numpy() turns the cluster column into a numpy array; "== c" then gives True or False for each complaint.
cluster_rows = []
for c in range(8):
    in_cluster = train["cluster"].to_numpy() == c
    # X[in_cluster] keeps the cluster's rows; mean(axis=0) averages each column (word).
    column_means = X[in_cluster].mean(axis=0)
    # The result is a 1 x words matrix; np.asarray(...).ravel() turns it into a flat list of numbers.
    cluster_rows.append(np.asarray(column_means).ravel())
# np.vstack stacks the 8 rows into one table: clusters x words.
cluster_words = np.vstack(cluster_rows)

for c in range(8):
    top_positions = np.argsort(-cluster_words[c])[:10]
    n_complaints = (train["cluster"] == c).sum()
    print(f"cluster {c} ({n_complaints} complaints):", " ".join(terms[top_positions]))

### 2.9 Compare themes and clusters

Rows are NMF themes; columns are clusters. A large number in a cell means the two methods put many of the same complaints together. Look for a theme that spreads over several clusters, and for a cluster that no theme matches.

In [ ]:
# Rows: NMF themes; columns: k-means clusters.
pd.crosstab(train["theme"], train["cluster"])

The representative complaints of a cluster are its own members that lie closest to its center. `kmeans.transform` gives each complaint's distance to every center. We keep only the complaints assigned to the cluster, then rank them by distance. We pick the cluster where the word *fee* has the largest average weight. It groups credit card charges, payments and fees, a group that no NMF theme has on its own.

In [ ]:
# The cluster where the word "fee" has the largest average weight.
fee_column = list(terms).index("fee")
fee_cluster = int(np.argmax(cluster_words[:, fee_column]))
print("Fee cluster:", fee_cluster)
# transform gives a table of distances (complaints x centers); keep the fee cluster's column.
distance = kmeans.transform(train_vectors)[:, fee_cluster]
# Keep the complaints assigned to the fee cluster: flatnonzero gives the row positions where the test is True.
members = np.flatnonzero(train["cluster"].to_numpy() == fee_cluster)
# Rank those members by distance and keep the 3 closest to the center.
closest = members[np.argsort(distance[members])[:3]]
for i in closest:
    print(f"{train.loc[i, 'complaint_id']} | {train.loc[i, 'product']} | distance {distance[i]:.3f}")
    print("   ", train.loc[i, "narrative"][:250], "\n")

### 2.10 Stability: refit NMF with another seed

Change only the seed and fit again. Then match each baseline theme with the theme of the new fit that shares the most top-10 terms. We call two themes a **match** when they share at least 7 of their top 10 terms. That cutoff is a choice; say which one you used when you report.

In [ ]:
nmf_seed1 = NMF(n_components=8, init="random", random_state=1, max_iter=1000)
W_seed1 = nmf_seed1.fit_transform(X)
train["theme_seed1"] = W_seed1.argmax(axis=1)
seed_match = match_themes(H, nmf_seed1.components_, terms)
# Count the baseline themes that share 7 or more of their top 10 terms with their best match.
n_matched = (seed_match["shared_terms"] >= 7).sum()
print("baseline themes with a match (7+ of 10 terms):", n_matched, "of 8")
seed_match[["theme", "best_match", "shared_terms", "top_terms", "match_terms"]]

Where did the complaints of the unmatched theme go? Each row below is a baseline theme, and each column is a theme of the seed-1 fit.

In [ ]:
# Rows: baseline themes (seed 0); columns: themes of the seed-1 fit.
pd.crosstab(train["theme"], train["theme_seed1"])

The seed-1 fit also has a theme that matches no baseline theme. Match in the other direction to find it, then read the openings of its strongest complaints.

In [ ]:
# Match in the other direction: each seed-1 theme against the baseline themes.
reverse = match_themes(nmf_seed1.components_, H, terms)
# idxmin gives the row with the fewest shared terms: the weakest match.
weakest_row = reverse["shared_terms"].idxmin()
new_theme = int(reverse.loc[weakest_row, "theme"])
n_new = (train["theme_seed1"] == new_theme).sum()
print("seed-1 theme with the weakest match:", new_theme, "| complaints:", n_new)
print("top terms:", reverse.loc[new_theme, "top_terms"])
# The openings of its three strongest complaints.
for i in np.argsort(-W_seed1[:, new_theme])[:3]:
    print("   ", train.loc[i, "complaint_id"], train.loc[i, "narrative"][:120])

The new theme is one template letter, copied with a different company name each time. Thinning kept these copies because their first words differ. A different seed gave this small group its own theme, while the theme about unauthorized card charges and disputes was absorbed elsewhere. Neither fit is wrong. A theme that appears in only one fit needs more reading before anyone acts on it.

### Exercise 1: a theme evidence table

Label the eight baseline themes. For each theme, read the top terms (2.4) and the three representative complaints in full (`show_theme(k, chars=None)`), then write a short label a manager would understand. Use a label such as "form letter" or "general story, no single problem" when that is what you find.

Then inspect two more complaints assigned to the theme, for example `train[train["theme"] == k].sample(2, random_state=0)`. Record the IDs of any that do not fit your label.

Make a table `evidence` with one row per theme and these columns:
- `theme`, `label` (yours), `complaints` (how many complaints have it as dominant theme, out of 6,000);
- `top_terms` (the top 8 terms);
- `examples`: the IDs of the 3 representative complaints (highest weight);
- `actionable`: `yes` or `no`, your judgment whether a manager could act on this theme as one issue;
- `counter_examples`: the IDs of the extra complaints that do not fit your label (an empty text `""` if both fit).

Print the table. Hint: start from `theme_table`, and get the example IDs with `train.loc[np.argsort(-W[:, k])[:3], "complaint_id"].tolist()`.

In [ ]:
# Exercise 1: your code here
# TODO: read show_theme(k, chars=None) for each theme k from 0 to 7, then write a label and "yes" or "no" for actionable.
# TODO: read two more complaints of each theme: train[train["theme"] == k].sample(2, random_state=0);
#       note the IDs of any that do not fit your label.
# TODO: build the table `evidence` with the columns theme, label, complaints, top_terms (top 8 terms),
#       examples (IDs of the 3 representative complaints), actionable and counter_examples.
# TODO: print the table.

### Exercise 2: another number of themes

Fit NMF again with **12 themes** (keep `init="random"`, `random_state=0`, `max_iter=1000`). Use new names: `nmf12`, `W12`, `H12` and the column `train["theme12"]`. Keep the baseline variables `nmf`, `W`, `H` and `train["theme"]` unchanged, because later steps and the Challenge compare with them. Then:
1. Print the top 10 terms of the 12 new themes.
2. Use `match_themes(H, ..., terms)` to match the 8 baseline themes with the 12 new ones. Print how many baseline themes have a match (7 or more shared terms), out of 8.
3. Pick one baseline theme without a match. Use `pd.crosstab` to show where its complaints went in the 12-theme fit, and say in a comment whether it split, merged or turned into something else. If every baseline theme has a match, use the theme with the fewest shared terms instead.

In [ ]:
# Exercise 2: your code here
# TODO: fit NMF with n_components=12 (init="random", random_state=0, max_iter=1000) on X.
#       Name the results nmf12, W12, H12 and train["theme12"]; do not overwrite nmf, W, H or train["theme"].
# TODO: print the top 10 terms of each of the 12 themes (use top_terms).
# TODO: run match_themes(H, <12-theme H>, terms) and print how many baseline themes share 7+ terms.
# TODO: for one unmatched baseline theme (or the weakest match), show pd.crosstab of baseline theme vs 12-theme fit;
#       say in a comment whether it split, merged or turned into something else.

### Exercise 3: sample counts are not prevalence

Choose one theme that you marked `actionable` in Exercise 1. Then:
1. Print its count and share in this sample: "k of 6,000".
2. Print its share **within each product**. Hint: `pd.crosstab(train["product"], train["theme"], normalize="index")`.
3. The course dataset has a fixed number of rows per product. Compute the share the theme would have if every product had the **same** number of complaints: the plain average of the six within-product shares. Compare it with the sample share.
4. Write one qualified sentence in a comment. Give the count with its denominator and name the sample. Say why the number is not the share of all consumers with this problem, and why it is not evidence of a cause.

In [ ]:
# Exercise 3: your code here
# TODO: pick one actionable theme and print its count and share as "k of 6,000".
# TODO: print its share within each product (pd.crosstab with normalize="index").
# TODO: average the six within-product shares (equal-weight share) and compare it with the sample share.
# TODO: write one qualified sentence in a comment (count with denominator, sample named, no prevalence or cause claim).

### Practice interpretation

Write 3–5 sentences:

- Which theme would you bring to a manager first, and why? Use your label, its count with the denominator, and one representative complaint ID.
- Which theme would you **not** report as an issue (for example a redaction, template or general-story theme), and what in the documents shows that?
- How stable were the themes (step 2.10 and Exercise 2)? Give your match counts and the cutoff.
- What can the theme counts not tell a manager? Use your result from Exercise 3.

**Your answer:**

### Submit

Run the whole notebook from the top (**Restart**, then **Run All**) and save it, so the outputs are saved in the file. Then commit and push it in GitHub Desktop before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Is k-means stable across seeds?

The **adjusted Rand index** compares two groupings of the same documents: 1 means identical, 0 means no more alike than chance. Compare k-means with seed 0 and seed 1, and compare k-means with the NMF themes.

In [ ]:
from sklearn.metrics import adjusted_rand_score
# Same k-means, seed 1; adjusted_rand_score(a, b) compares two lists of group numbers.
kmeans_seed1 = KMeans(n_clusters=8, n_init=10, random_state=1).fit(train_vectors)
print("k-means seed 0 vs seed 1:", round(adjusted_rand_score(train["cluster"], kmeans_seed1.labels_), 2))
print("NMF seed 0 vs seed 1:   ", round(adjusted_rand_score(train["theme"], train["theme_seed1"]), 2))
print("NMF themes vs k-means:  ", round(adjusted_rand_score(train["theme"], train["cluster"]), 2))

### B. A second topic-model family: LDA

**LDA** (latent Dirichlet allocation) is a probabilistic topic model. It works on word counts instead of TF-IDF. Compare its top terms with your NMF themes. This takes longer than NMF (up to a minute).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
# CountVectorizer counts words instead of weighting them; the other settings are the same as in 2.3.
counter = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]{2,}\b", min_df=5, max_df=0.5)
counts = counter.fit_transform(train["theme_text"])
count_terms = counter.get_feature_names_out()
# learning_method="batch" uses all complaints in each of the max_iter=10 rounds.
lda = LatentDirichletAllocation(n_components=8, random_state=0, learning_method="batch", max_iter=10).fit(counts)
# components_ plays the role of H, so top_terms works here too.
for k, words in enumerate(top_terms(lda.components_, count_terms)):
    print(f"LDA topic {k}:", " ".join(words))

### C. Themes in unseen complaints

The fitted vectorizer and NMF model can **transform** new text without refitting. Apply them to the `valid` split and compare theme shares. Similar shares suggest the themes are not specific to the training complaints. The product mix of `valid` was fixed the same way, so this is still not prevalence.

In [ ]:
valid = complaints[complaints["split"].eq("valid")].reset_index(drop=True)
# Clean the text the same way, then transform (not fit): the train vocabulary and themes are reused.
valid_text = valid["narrative"].str.replace(REDACTION, " ", regex=True)
X_valid = vectorizer.transform(valid_text)
valid_theme = nmf.transform(X_valid).argmax(axis=1)
# Share of complaints per theme: normalize=True gives shares instead of counts; sort_index orders by theme.
train_share = train["theme"].value_counts(normalize=True).sort_index().round(3)
valid_share = pd.Series(valid_theme).value_counts(normalize=True).sort_index().round(3)
pd.DataFrame({"train_share": train_share, "valid_share": valid_share})

## Further reading

- [scikit-learn user guide: Non-negative matrix factorization (NMF)](https://scikit-learn.org/stable/modules/decomposition.html#non-negative-matrix-factorization-nmf-or-nnmf)
- [scikit-learn example: Topic extraction with NMF and LDA](https://scikit-learn.org/stable/auto_examples/applications/plot_topics_extraction_with_nmf_lda.html)
- [Jurafsky and Martin, Speech and Language Processing, chapter on vector semantics and embeddings](https://web.stanford.edu/~jurafsky/slp3/)
- [BERTopic documentation](https://maartengr.github.io/BERTopic/): a popular pipeline that clusters embeddings. It needs extra packages and is not required.

Next, Module 11 goes inside each complaint. It extracts specific facts, such as an amount or a requested remedy, into a written schema with rules and saved LLM outputs.